In [ ]:
import os
import marimo as mo
import deepcor

os.chdir(mo.notebook_dir())  # all paths relative to the notebook

In [ ]:
# GPU check (optional)
deepcor.utils.check_gpu_and_speedup(tensor_size=(1000, 1000), n_iter=100)

In [ ]:
# ---- Data paths (the only thing you normally edit) ----
bids_path = "../Data/fMRI-Data/studyforrest-fmriprep/"

subs = sorted(s for s in os.listdir(bids_path) if s.startswith("sub-"))
s, r = 0, 4  # subject index, run
sub_id, run = subs[s], str(r)

session, task = "ses-localizer", "objectcategories"
space = "MNI152NLin2009cAsym"
base = os.path.join(bids_path, sub_id, session, "func")

epi_path = os.path.join(
    base, f"{sub_id}_{session}_task-{task}_run-{run}_bold_space-{space}_preproc.nii.gz"
)
confounds_path = os.path.join(
    base, f"{sub_id}_{session}_task-{task}_run-{run}_bold_confounds.tsv"
)
gm_mask_path = os.path.join(bids_path, "mask_roi.nii")
cf_mask_path = os.path.join(bids_path, "mask_roni.nii")

output_dir = os.path.join(
    "../Data/DeepCor-Outputs", "forrest-simple-v2", f"S{s}-R{r}-cvae_v2"
)

for p in (epi_path, confounds_path, gm_mask_path, cf_mask_path):
    assert os.path.exists(p), f"missing: {p}"
print("EPI:", epi_path)
print("output_dir:", output_dir)

EPI: ../Data/fMRI-Data/studyforrest-fmriprep/sub-01/ses-localizer/func/sub-01_ses-localizer_task-objectcategories_run-4_bold_space-MNI152NLin2009cAsym_preproc.nii.gz
output_dir: ../Data/DeepCor-Outputs/forrest-simple-v2/S0-R4-cvae_v2


In [ ]:
# ---- Train + denoise ----
# Config is optional; omit it for sensible defaults. To customise, pass e.g.
#   config = deepcor.DeepCorConfig()
#   config.training.n_epochs = 100
#   denoiser = deepcor.DeepCor(model_version="v2", config=config)
#
# No `dashboard`/`on_epoch` => nothing is plotted interactively; the training
# dashboard is still saved to output_dir every epoch. A per-epoch progress
# line (subject/run, started, elapsed, ETA) is printed below.
denoiser = deepcor.DeepCor(model_version="v2")
denoiser.config.training.n_epochs = 5
denoiser.config.training.n_repetitions = 5

result = denoiser.fit_denoise(
    epi_path,
    gm_mask_path,
    cf_mask_path,
    confounds_path,
    output_dir,
    subject_idx=s,
    run_idx=r,
)

device is cuda:0
model_version is v2
Preparing training data...
obs_list_coords.shape: (41147, 4, 156)
noi_list_coords.shape: (9105, 4, 156)
upsampling noi_list_coords
obs_list_coords.shape: (41147, 4, 156)
noi_list_coords.shape: (41147, 4, 156)
S0R4 Training started at: 2026-06-17 12:35:29, elapsed: 0:00:52.6 Ens:1/5, E:5/5 ETA:0:03:30.4
S0R4 Training started at: 2026-06-17 12:35:29, elapsed: 0:01:57.6 Ens:2/5, E:5/5 ETA:0:02:56.5
S0R4 Training started at: 2026-06-17 12:35:29, elapsed: 0:03:01.8 Ens:3/5, E:5/5 ETA:0:02:01.2
S0R4 Training started at: 2026-06-17 12:35:29, elapsed: 0:03:23.9 Ens:4/5, E:1/5 ETA:0:01:54.7

interruption: This cell was interrupted and needs to be re-run

In [ ]:
print("Denoised:", result.denoised_path)
print("Preproc :", result.preproc_path)
print("CompCor :", result.compcor_path)
result.denoised_path

Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)